In [ ]:
# 1. Install necessary libraries
!pip install flickrapi folium pandas

import flickrapi
import pandas as pd
import folium
import time
from google.colab import files
from IPython.display import display

# 2. Enter your Flickr API credentials (replace with your actual Key and Secret)
API_KEY = ''
API_SECRET = ''

# Initialize Flickr API
flickr = flickrapi.FlickrAPI(API_KEY, API_SECRET, format='parsed-json')

# 3. Set search parameters (Manchester city centre and surrounding main industrial areas)
# Bounding box: West to Salford Quays, East to Ancoats, covering major canal zones North-South
manchester_bbox = '-2.3200,53.4400,-2.1700,53.5100'

# Manchester industrial heritage related factory and building keywords
industrial_keywords = [
    'industrial heritage', 'cotton mill', 'textile mill',
    'spinning mill', 'warehouse', 'red brick',
    'factory', 'mill', 'ironworks', 'canal',
    'viaduct', 'cottonopolis', 'industrial revolution'
]
tags_string = ','.join(industrial_keywords)

print(f"Searching Flickr for the following keywords: {tags_string}")
print("Starting auto-pagination fetch, this may take some time...")

try:
    # 4. Use a loop for auto-pagination
    all_photos_dict = {}  # Use a dictionary to auto-remove duplicates by photo ID
    current_page = 1
    max_pages = 20        # Set safety limit to prevent infinite loops (20 pages * 250 = max 5000 photos)

    while current_page <= max_pages:
        print(f"Fetching data for page {current_page}...")

        photos = flickr.photos.search(
            tags=tags_string,
            tag_mode='any',       # Tag mode 'any' means any of the industrial keywords will match
            bbox=manchester_bbox, # Limit to Manchester area
            has_geo=1,
            extras='geo,url_s',
            per_page=250,         # Max per page limit for geo-search API
            page=current_page
        )

        photo_list = photos['photos']['photo']

        # If current page is empty, all data has been fetched
        if not photo_list:
            print("No more photos, fetching completed.")
            break

        # Add photos to dictionary (use ID to deduplicate)
        for photo in photo_list:
            all_photos_dict[photo['id']] = photo

        # Check if reached the last page returned by Flickr
        total_pages = photos['photos']['pages']
        if current_page >= total_pages:
            print(f"Reached the last page ({total_pages}), fetching completed.")
            break

        current_page += 1
        time.sleep(1) # Polite 1-second delay to prevent API ban

    # Convert dictionary back to list
    final_photo_list = list(all_photos_dict.values())
    print(f"\nSuccess! Fetched {len(final_photo_list)} unique photos related to Manchester industrial heritage!")

    # 5. Convert data to Pandas DataFrame and clean
    if len(final_photo_list) > 0:
        df = pd.DataFrame(final_photo_list)
        df['latitude'] = df['latitude'].astype(float)
        df['longitude'] = df['longitude'].astype(float)

        # Filter out invalid data with coordinates at 0
        df = df[(df['latitude'] != 0.0) & (df['longitude'] != 0.0)]

        # 6. Create pure dark interactive map with no labels
        print("Generating pure dark map with no labels...")
        # Set map center to Manchester City Centre
        manchester_map = folium.Map(
            location=[53.4794, -2.2453],
            zoom_start=13, # Zoom out slightly to overlook the entire industrial network
            tiles='https://{s}.basemaps.cartocdn.com/dark_nolabels/{z}/{x}/{y}{r}.png',
            attr='&copy; OpenStreetMap contributors &copy; CARTO'
        )

        # Add each photo as a semi-transparent blue solid dot to the map (no pop-up)
        for idx, row in df.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=1,
                color='blue',
                weight=1,
                fill=True,
                fill_color='blue',
                fill_opacity=0.7
            ).add_to(manchester_map)

        # 7. Export HTML and CSV files and auto-download
        print("Preparing to download files to your computer...")

        html_filename = 'manchester_industrial_heritage_map.html'
        manchester_map.save(html_filename)

        csv_filename = 'manchester_industrial_coordinates.csv'
        export_df = df[['latitude', 'longitude']]
        export_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

        files.download(html_filename)
        files.download(csv_filename)

        print("All done! Please check your browser's download pop-ups.")
        print("Map preview below:")

        display(manchester_map)

    else:
        print("No photos found matching these industrial keywords, please try expanding the geographic area or modifying keywords.")

except Exception as e:
    print(f"An error occurred: {e}")

Searching Flickr for the following keywords: industrial heritage,cotton mill,textile mill,spinning mill,warehouse,red brick,factory,mill,ironworks,canal,viaduct,cottonopolis,industrial revolution
Starting auto-pagination fetch, this may take some time...
Fetching data for page 1...
Fetching data for page 2...
Fetching data for page 3...
Fetching data for page 4...
Fetching data for page 5...
Fetching data for page 6...
Fetching data for page 7...
Fetching data for page 8...
Fetching data for page 9...
Fetching data for page 10...
Fetching data for page 11...
Fetching data for page 12...
Fetching data for page 13...
Fetching data for page 14...
Fetching data for page 15...
Fetching data for page 16...
Fetching data for page 17...
Fetching data for page 18...
Fetching data for page 19...
Reached the last page (19), fetching completed.

Success! Fetched 3670 unique photos related to Manchester industrial heritage!
Generating pure dark map with no labels...
Preparing to download files to y

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All done! Please check your browser's download pop-ups.
Map preview below:
